Install Dependencies

In [111]:
!pip install crewai
!pip install crewai-tools
!pip install langchain-groq
!pip install groq
!pip install python-dotenv
!pip install streamlit
!pip install requests
!pip install pillow
!pip install apscheduler
!pip install pandas
!pip install sqlalchemy
!pip install streamlit_chat


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.0 MB/s eta 0:00:00


In [32]:
GROQ_API_KEY='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'

In [49]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(
    api_key=GROQ_API_KEY, # Using the GROQ_API_KEY variable
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    max_tokens=4096
)

In [80]:
#idea_agent.py
from crewai import Agent

# Using the ChatGroq instance defined in Qfbv7ve_OB2g as a dictionary

idea_agent = Agent(
    role="AI Marketing Strategist",

    goal="""
    Transform a simple business idea into a complete marketing strategy,
    target audience analysis, campaign objectives,
    content pillars and posting schedule.
    """,

    backstory="""
    You are a world-class marketing strategist with expertise in branding,
    digital marketing, real estate marketing, social media campaigns,
    customer psychology and business growth.
    """,

    llm=llm.model_dump(), # Using the ChatGroq instance directly

    allow_delegation=False,

    verbose=True
)

In [81]:
# content_agent.py
from crewai import Agent

# llm is already defined in the global scope from cell Qfbv7ve_OB2g

content_agent = Agent(
    role="Content Creator",

    goal="Create engaging social media content.",

    backstory="Expert copywriter.",

    llm=llm.model_dump(), # Using the ChatGroq instance directly

    verbose=True
)

In [82]:
# editor_agent.py
from crewai import Agent

# llm is already defined in the global scope from cell Qfbv7ve_OB2g

editor_agent = Agent(
    role="Content Editor",

    goal="Improve grammar, tone and SEO.",

    backstory="Professional editor.",

    llm=llm.model_dump(), # Using the ChatGroq instance directly

    verbose=True
)

In [83]:
# flyer_agent.py
from crewai import Agent

# llm is already defined in the global scope from cell Qfbv7ve_OB2g

flyer_agent = Agent(
    role="Creative Director",

    goal="""
    Produce detailed prompts for AI flyer generation.S
    """,

    backstory="Award-winning creative designer.",

    llm=llm.model_dump(), # Using the ChatGroq instance directly

    verbose=True
)

In [84]:
# scheduler_agent.py
from crewai import Agent

# llm is already defined in the global scope from cell Qfbv7ve_OB2g

scheduler_agent = Agent(
    role="Campaign Scheduler",
    goal="Create a detailed publishing schedule for marketing campaigns.",
    backstory="Expert in optimizing content delivery across platforms, ensuring maximum reach and engagement.",
    llm=llm.model_dump(), # Using the ChatGroq instance directly
    verbose=True
)

In [66]:
# scheduler_task.py
from crewai import Task


scheduler_task = Task(

    description="""
    Based on the completed marketing campaign,

    determine:

    1. Best platforms

    2. Best posting time

    3. Posting frequency

    4. Campaign duration

    5. Posting calendar

    6. Suggested hashtags

    7. Audience demographics

    8. Geographic targeting

    9. Budget recommendation

    10. KPI metrics

    Campaign Information:

    {campaign}
    """,

    expected_output="""
    A complete publishing schedule including
    platforms, dates, times, frequency,
    hashtags and performance KPIs.
    """,

    agent=scheduler_agent
)

In [58]:
# tasks.py
from crewai import Task

# The agents are already defined in previous cells and are in the global scope.
# Therefore, explicit imports from 'agents' directory are not needed.
# from agents.idea_agent import idea_agent
# from agents.content_agent import content_agent
# from agents.editor_agent import editor_agent
# from agents.flyer_agent import flyer_agent

idea_task = Task(
    description="""
    Analyze the following business idea:

    {idea}

    Produce a complete marketing strategy.
    """,

    expected_output="Complete marketing strategy.",

    agent=idea_agent
)

content_task = Task(
    description="""
    Using the strategy,

    write content for:

    Facebook

    Instagram

    LinkedIn

    X

    WhatsApp
    """,

    expected_output="Optimized social media posts.",

    agent=content_agent
)

editor_task = Task(
    description="""
    Review all generated content.

    Improve grammar, SEO and readability.
    """,

    expected_output="Edited content.",

    agent=editor_agent
)

flyer_task = Task(
    description="""
    Produce a professional flyer prompt
    suitable for an AI image generator.
    """,

    expected_output="Detailed flyer prompt.",

    agent=flyer_agent
)

In [107]:
# main.py
from crewai import Crew

# The agents and tasks are defined in previous cells and are globally available.
# Explicit imports from 'agents' or 'tasks' modules are not needed.
# from agents.idea_agent import idea_agent
# from agents.content_agent import content_agent
# from agents.editor_agent import editor_agent
# from agents.flyer_agent import flyer_agent

# from tasks import (
#     idea_task,
#     content_task,
#     editor_task,
#     flyer_task
# )

crew = Crew(

    agents=[
        idea_agent,
        content_agent,
        editor_agent,
        flyer_agent
    ],

    tasks=[
        idea_task,
        content_task,
        editor_task,
        flyer_task
    ],

    verbose=True
)


print()

Streamlit app

app.py

In [129]:
import streamlit as st
from groq import Groq

app_py_content = """
import streamlit as st
from groq import Groq
from crewai import Crew
from crewai import Agent, Task

# Assuming GROQ_API_KEY is available as an environment variable or set globally
# In a production Streamlit app, you would use st.secrets for better security
# but for a Colab environment where it's globally defined, direct access is fine.
# Ensure GROQ_API_KEY is defined before running this app.
GROQ_API_KEY = "" # Replace with actual key or ensure it's set in environment

st.set_page_config(
    page_title="BrandPilot AI",
    page_icon="🚀",
    layout="wide"
)

st.title("🚀 BrandPilot AI")
st.subheader("Your AI Marketing Assistant")

client = Groq(api_key=GROQ_API_KEY)

idea = st.text_area(
    "Enter your marketing idea",
    placeholder="Example: Promote luxury plots at Vannahills Estate with flexible payment plans."
)

if st.button("Generate Campaign"):

    if not idea.strip():
        st.warning("Please enter a marketing idea.")
    else:

        with st.spinner("Generating campaign..."):

            prompt = f'''
You are an expert marketing strategist.

Generate:

1. Marketing Strategy
2. Target Audience
3. Facebook Post
4. Instagram Caption
5. LinkedIn Post
6. Five Hashtags
7. Call To Action

Idea:

{idea}
'''

            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            )

            result = response.choices[0].message.content

        st.success("Campaign Generated Successfully!")

        st.markdown(result)
"""

with open("app.py", "w") as f:
    f.write(app_py_content)

print("app.py has been created successfully!")

app.py has been created successfully!


requirement.txt

In [119]:
# requirement
import pkg_resources
installed_packages = {pkg.key: pkg.version for pkg in pkg_resources.working_set}
required_packages = [
    'crewai',
    'crewai-tools',
    'langchain-groq',
    'groq',
    'python-dotenv',
    'streamlit',
    'requests',
    'pillow',
    'apscheduler',
    'pandas',
    'sqlalchemy',
    'streamlit-chat',
]

# Create the requirements.txt content
requirements_content = []
for pkg in required_packages:
    if pkg in installed_packages:
        requirements_content.append(f"{pkg}=={installed_packages[pkg]}")
    else:
        print(f"Warning: Package '{pkg}' not found in installed packages. Please ensure it's installed.")

# Write to requirements.txt file
with open("requirements.txt", "w") as f:
    f.write("\n".join(requirements_content))

print("requirements.txt has been created successfully!")



requirements.txt has been created successfully!


/tmp/ipykernel_4447/829986887.py:2: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


In [120]:
gitignore_content = """
.streamlit/
__pycache__/
*.pyc
.env
"""

with open(".gitignore", "w") as f:
    f.write(gitignore_content)

print(".gitignore file created successfully.")

.gitignore file created successfully.


In [127]:
# config.py
import streamlit as st
import os

# ==========================================
# AI CONFIGURATION
# ==========================================

# Use the globally defined GROQ_API_KEY
GROQ_API_KEY = GROQ_API_KEY

MODEL_NAME = os.getenv(
    "MODEL_NAME",
    "llama-3.3-70b-versatile"
)

TEMPERATURE = float(
    os.getenv("TEMPERATURE", 0.7)
)

MAX_TOKENS = int(
    os.getenv("MAX_TOKENS", 4096)
)

# ==========================================
# DATABASE
# ==========================================

DATABASE_URL = os.getenv(
    "DATABASE_URL",
    "sqlite:///brandpilot.db"
)

# ==========================================
# SOCIAL MEDIA TOKENS
# ==========================================

FACEBOOK_ACCESS_TOKEN = os.getenv(
    "FACEBOOK_ACCESS_TOKEN", ""
)

INSTAGRAM_ACCESS_TOKEN = os.getenv(
    "INSTAGRAM_ACCESS_TOKEN", ""
)

LINKEDIN_ACCESS_TOKEN = os.getenv(
    "LINKEDIN_ACCESS_TOKEN", ""
)

X_ACCESS_TOKEN = os.getenv(
    "X_ACCESS_TOKEN", ""
)

# ==========================================
# IMAGE GENERATION
# ==========================================

# Assuming HF_TOKEN might be HUGGINGFACEHUB_API_TOKEN from the context
HF_TOKEN = os.getenv(
    "HF_TOKEN", HUGGINGFACEHUB_API_TOKEN if 'HUGGINGFACEHUB_API_TOKEN' in globals() else ""
)

# ==========================================
# APPLICATION SETTINGS
# ==========================================

APP_NAME = "BrandPilot AI"

APP_VERSION = "1.0.0"

DEBUG = os.getenv(
    "DEBUG",
    "True"
).lower() == "true" # Convert string 'True'/'False' to boolean

In [130]:
import os
import streamlit as st
from dotenv import load_dotenv

load_dotenv()

def get_secret(key, default=None):
    """Read from Streamlit secrets first, then fall back to environment variables."""
    try:
        return st.secrets[key]
    except Exception:
        return os.getenv(key, default)

# AI
GROQ_API_KEY = get_secret("GROQ_API_KEY")
MODEL_NAME = get_secret("MODEL_NAME", "llama-3.3-70b-versatile")
TEMPERATURE = float(get_secret("TEMPERATURE", 0.7))
MAX_TOKENS = int(get_secret("MAX_TOKENS", 4096))

# Database
DATABASE_URL = get_secret("DATABASE_URL", "sqlite:///brandpilot.db")

# Social Media
FACEBOOK_ACCESS_TOKEN = get_secret("FACEBOOK_ACCESS_TOKEN", "")
INSTAGRAM_ACCESS_TOKEN = get_secret("INSTAGRAM_ACCESS_TOKEN", "")
LINKEDIN_ACCESS_TOKEN = get_secret("LINKEDIN_ACCESS_TOKEN", "")
X_ACCESS_TOKEN = get_secret("X_ACCESS_TOKEN", "")

# Hugging Face
HF_TOKEN = get_secret("HF_TOKEN", "")

# App
APP_NAME = "BrandPilot AI"
APP_VERSION = "1.0.0"
DEBUG = str(get_secret("DEBUG", "True")).lower() == "true"

In [137]:
config_py_content = '''
import os
import streamlit as st
from dotenv import load_dotenv

load_dotenv()

def get_secret(key, default=None):
    """Read from Streamlit secrets first, then fall back to environment variables."""
    try:
        return st.secrets[key]
    except Exception:
        return os.getenv(key, default)

# AI
GROQ_API_KEY = get_secret("GROQ_API_KEY")
MODEL_NAME = get_secret("MODEL_NAME", "llama-3.3-70b-versatile")
TEMPERATURE = float(get_secret("TEMPERATURE", 0.7))
MAX_TOKENS = int(get_secret("MAX_TOKENS", 4096))

# Database
DATABASE_URL = get_secret("DATABASE_URL", "sqlite:///brandpilot.db")

# Social Media
FACEBOOK_ACCESS_TOKEN = get_secret("FACEBOOK_ACCESS_TOKEN", "")
INSTAGRAM_ACCESS_TOKEN = get_secret("INSTAGRAM_ACCESS_TOKEN", "")
LINKEDIN_ACCESS_TOKEN = get_secret("LINKEDIN_ACCESS_TOKEN", "")
X_ACCESS_TOKEN = get_secret("X_ACCESS_TOKEN", "")

# Hugging Face
HF_TOKEN = get_secret("HF_TOKEN", "")

# App
APP_NAME = "BrandPilot AI"
APP_VERSION = "1.0.0"
DEBUG = str(get_secret("DEBUG", "True")).lower() == "true"
'''

with open("config.py", "w") as f:
    f.write(config_py_content)

print("config.py has been created successfully!")

config.py has been created successfully!


In [138]:
config_py_content = '''
import os
import streamlit as st
from dotenv import load_dotenv

load_dotenv()

def get_secret(key, default=None):
    """Read from Streamlit secrets first, then fall back to environment variables."""
    try:
        return st.secrets[key]
    except Exception:
        return os.getenv(key, default)

# AI
GROQ_API_KEY = get_secret("GROQ_API_KEY")
MODEL_NAME = get_secret("MODEL_NAME", "llama-3.3-70b-versatile")
TEMPERATURE = float(get_secret("TEMPERATURE", 0.7))
MAX_TOKENS = int(get_secret("MAX_TOKENS", 4096))

# Database
DATABASE_URL = get_secret("DATABASE_URL", "sqlite:///brandpilot.db")

# Social Media
FACEBOOK_ACCESS_TOKEN = get_secret("FACEBOOK_ACCESS_TOKEN", "")
INSTAGRAM_ACCESS_TOKEN = get_secret("INSTAGRAM_ACCESS_TOKEN", "")
LINKEDIN_ACCESS_TOKEN = get_secret("LINKEDIN_ACCESS_TOKEN", "")
X_ACCESS_TOKEN = get_secret("X_ACCESS_TOKEN", "")

# Hugging Face
HF_TOKEN = get_secret("HF_TOKEN", "")

# App
APP_NAME = "BrandPilot AI"
APP_VERSION = "1.0.0"
DEBUG = str(get_secret("DEBUG", "True")).lower() == "true"
'''

with open("config.py", "w") as f:
    f.write(config_py_content)

print("config.py has been created successfully!")

config.py has been created successfully!


### Next Steps:

1.  **Add NGROK_AUTH_TOKEN to Colab Secrets**: The Streamlit app requires `ngrok` to create a public URL. You need to provide your `NGROK_AUTH_TOKEN` in Colab's secrets manager. Click the "🔑 Secrets" icon on the left sidebar, add a new secret named `NGROK_AUTH_TOKEN`, and paste your ngrok authentication token there.

2.  **Re-run the Streamlit Launch Cell**: After adding the token, please re-run the last code cell (`9655bf4f`) that starts Streamlit with ngrok. This will now successfully create a public URL for your application.

In [132]:
# Update app.py content to use config.py
app_py_content_updated = """
import streamlit as st
from groq import Groq
from crewai import Crew
from crewai import Agent, Task
import config # Import the config file

# Use GROQ_API_KEY from config.py
GROQ_API_KEY = config.GROQ_API_KEY

st.set_page_config(
    page_title=config.APP_NAME,
    page_icon="🚀",
    layout="wide"
)

st.title(f"🚀 {config.APP_NAME}")
st.subheader("Your AI Marketing Assistant")

client = Groq(api_key=GROQ_API_KEY)

idea = st.text_area(
    "Enter your marketing idea",
    placeholder="Example: Promote luxury plots at Vannahills Estate with flexible payment plans."
)

if st.button("Generate Campaign"):

    if not idea.strip():
        st.warning("Please enter a marketing idea.")
    else:

        with st.spinner("Generating campaign..."):

            prompt = f'''
You are an expert marketing strategist.

Generate:

1. Marketing Strategy
2. Target Audience
3. Facebook Post
4. Instagram Caption
5. LinkedIn Post
6. Five Hashtags
7. Call To Action

Idea:

{idea}
'''

            response = client.chat.completions.create(
                model=config.MODEL_NAME,
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            )

            result = response.choices[0].message.content

        st.success("Campaign Generated Successfully!")

        st.markdown(result)
"""

with open("app.py", "w") as f:
    f.write(app_py_content_updated)

print("app.py has been updated successfully to use config.py!")

app.py has been updated successfully to use config.py!


Your model's integration within the Streamlit app is set up correctly in `app.py`, and `config.py` has also been created. The core logic for generating campaigns using the Groq model (`llama-3.3-70b-versatile`) is in place.

However, for actual deployment and to test it via a public URL, you still need to provide your `NGROK_AUTH_TOKEN` in Colab's secrets. Without it, `ngrok` cannot create the tunnel necessary to expose your Streamlit application.

**To get your app ready for testing/deployment:**

1.  **Add `NGROK_AUTH_TOKEN`**: Go to the "🔑 Secrets" icon on the left sidebar in Colab. Add a new secret named `NGROK_AUTH_TOKEN` and paste your ngrok authentication token. You can find this token on your ngrok dashboard.
2.  **Re-run the Streamlit Launch Cell**: Once the token is added, please re-execute cell `9655bf4f`. This will then successfully launch your Streamlit application and provide you with a public URL.

In [133]:
%%bash
pip install -q pyngrok

In [144]:
from pyngrok import ngrok
import os

# Terminate any previous ngrok tunnels
ngrok.kill()

# Get the ngrok auth token from environment variables or Colab secrets
# You might need to set this in your Colab secrets if it's not already there
# Go to the 'Secrets' tab (lock icon) in the left sidebar and add NGROK_AUTH_TOKEN
NGROK_AUTH_TOKEN = os.getenv("NGROK_AUTH_TOKEN") # Correctly retrieve the token by its name

if not NGROK_AUTH_TOKEN:
    print("NGROK_AUTH_TOKEN not found. Please add it to your Colab secrets.")
else:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

    # Open a ngrok tunnel to the Streamlit port (default 8501)
    public_url = ngrok.connect(8501)
    print(f"Streamlit App URL: {public_url}")

    # Run the Streamlit app
    # This command will block the cell execution until Streamlit is stopped
    !streamlit run app.py

NGROK_AUTH_TOKEN not found. Please add it to your Colab secrets.
